## 01 · Exploración y preparación del corpus

**Guía de ejecución:** Entrada: data/raw/chatgpt_results_final_language_filtered.csv. Produce archivos en data/processed y muestras en data/pilot. Necesita el CSV original de Visual Studio.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv((PROJECT_ROOT / 'data' / 'raw' / 'chatgpt_results_final_language_filtered.csv'))

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
summary = (
    df["detected_language_final"]
    .value_counts()
    .rename("count")
    .to_frame()
)

summary["percentage"] = (summary["count"] / len(df) * 100).round(2)

print(summary)

In [ ]:
df_english = df[df["detected_language_final"] == "English"]

In [ ]:
# Número de conversaciones de ChatGPT en inglés (para un total de 633783 registros)
df_english["url"].nunique()

In [ ]:
ensure_output_dir(PROCESSED_DIR)
# Guardar en formato Parquet
df_english.to_parquet(
    (PROJECT_ROOT / 'data' / 'processed' / 'chatgpt_conversations_english.parquet'),
    index=False
)

In [ ]:
# Leer el archivo Parquet
df_english = pd.read_parquet(
    (PROJECT_ROOT / 'data' / 'processed' / 'chatgpt_conversations_english.parquet')
)

In [ ]:
df_english.info()

### Construcción del DataFrame a nivel conversación

In [ ]:
# 1. Configuración
PILOT_SIZE = 200
RANDOM_STATE = 42
REDACTED_TOKEN = "<REDACTED>"

RELEVANT_COLUMNS = [
    "url",
    "turns_count",
    "message_index",
    "role",
    "plain_text",
    "topic"
]

In [ ]:
# 2. Selección de columnas
df_work = df_english[RELEVANT_COLUMNS].copy()

In [ ]:
# 3. Eliminar conversaciones en las que al menos un mensaje contiene el token <REDACTED>
redacted_urls = (
    df_work.loc[
        df_work["plain_text"]
        .fillna("")
        .str.contains(REDACTED_TOKEN, regex=False),
        "url"
    ]
    .unique()
)

print(f"Conversaciones con <REDACTED>: {len(redacted_urls):,}")

In [ ]:
print(f"Conversaciones en total: {len(df_work["url"].unique())}")

In [ ]:
# Eliminamos completamente las conversaciones anteriores
df_work = df_work[
    ~df_work["url"].isin(redacted_urls)
].copy()

In [ ]:
len(df_work)

In [ ]:
# 4. Eliminar mensajes sin texto

# Una vez eliminadas las conversaciones con información redactada,
# descartamos mensajes que no contienen texto.
df_work = df_work[
    df_work["plain_text"].notna()
].copy()

# Opcional: eliminar mensajes vacíos o compuestos únicamente por espacios.
df_work["plain_text"] = df_work["plain_text"].str.strip()

df_work = df_work[
    df_work["plain_text"] != ""
].copy()

In [ ]:
len(df_work)

In [ ]:
# 5. Ordenar los mensajes de cada conversación
df_work = df_work.sort_values(
    ["url", "message_index"]
).reset_index(drop=True)

In [ ]:
# 6. Construir representación textual de la conversación

def build_conversation_text(group):
    """
    Convierte todos los mensajes de una conversación en una representación
    textual numerada que será enviada posteriormente a los LLM anotadores.
    """

    lines = []

    # Nos aseguramos nuevamente de mantener el orden original
    group = group.sort_values("message_index")

    for turn_number, (_, row) in enumerate(group.iterrows(), start=1):

        role = str(row["role"]).strip().upper()
        text = row["plain_text"]

        lines.append(
            f"[TURN {turn_number} | {role}]\n{text}"
        )

    return "\n\n".join(lines)


In [ ]:
# 7. Transformación del dataset de mensajes a conversaciones

conversation_records = []

for url, group in df_work.groupby("url", sort=False):

    group = group.sort_values("message_index")

    topics = (
        group["topic"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    conversation_text = build_conversation_text(group)

    # Filtro para asegurarnos de que las conversaciones están formadas por usuario-agente
    # De esta forma también eliminaremos aquellas conversaciones de un solo mensaje
    roles = set(group["role"])

    has_user = "user" in roles
    has_assistant = "llm" in roles

    conversation_records.append({
        "url": url,

        # turns_count proporcionado originalmente por SHARECHAT
        "turns_count_original": group["turns_count"].iloc[0],

        # Número real de mensajes conservados por nosotros
        "n_messages": len(group),

        # Número de temas diferentes presentes en los mensajes
        "n_topics": len(topics),

        # Lista de temas presentes
        "topics": topics,

        # Texto completo preparado para la anotación
        "conversation_text": conversation_text,

        # Longitud aproximada del contexto
        "char_count": len(conversation_text),

        # Presencia de roles
        "has_user": has_user,
        "has_assistant": has_assistant
    })


conversation_df = pd.DataFrame(conversation_records)

In [ ]:
pd.crosstab(
    conversation_df["has_user"],
    conversation_df["has_assistant"]
)

In [ ]:
# 8. Filtro para mentener conversaciones con al menos una intervención humana y una invertención de un LLM
conversation_df = conversation_df[
    conversation_df["has_user"] &
    conversation_df["has_assistant"]
].copy()

In [ ]:
# 9. Controles de calidad básicos
print("\n--- Dataset conversacional ---")
print(f"Conversaciones disponibles: {len(conversation_df):,}")

print("\nDistribución del número de mensajes:")
print(
    conversation_df["n_messages"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
)

print("\nDistribución del número de caracteres:")
print(
    conversation_df["char_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
)


In [ ]:
# 10. Crear estratos de longitud
def assign_length_bin(n_messages):
    if n_messages <= 2:
        return "very_short"
    elif n_messages <= 4:
        return "short"
    elif n_messages <= 12:
        return "medium"
    else:
        return "long"


conversation_df["length_bin"] = (
    conversation_df["n_messages"]
    .apply(assign_length_bin)
)

# Contamos cuántas tenemos de cada grupo
conversation_df["length_bin"].value_counts()

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Orden lógico de los estratos
order = ["very_short", "short", "medium", "long"]

# Contar conversaciones por estrato y respetar el orden
counts = (
    conversation_df["length_bin"]
    .value_counts()
    .reindex(order, fill_value=0)
)

# Gráfico
plt.figure(figsize=(8, 5))
counts.plot(kind="bar")

plt.title("Distribución de conversaciones según su longitud")
plt.xlabel("Estrato de longitud")
plt.ylabel("Número de conversaciones")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
%pip install seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

order = ["very_short", "short", "medium", "long"]

plt.figure(figsize=(8, 5))

sns.countplot(
    data=conversation_df,
    x="length_bin",
    order=order
)

plt.title("Distribución de conversaciones según su longitud")
plt.xlabel("Estrato de longitud")
plt.ylabel("Número de conversaciones")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo general
sns.set_theme(style="whitegrid", context="notebook")

order = ["very_short", "short", "medium", "long"]

counts = (
    conversation_df["length_bin"]
    .value_counts()
    .reindex(order, fill_value=0)
)

# Etiquetas algo más legibles
labels = ["Muy cortas", "Cortas", "Medias", "Largas"]

fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    labels,
    counts.values,
    color=sns.color_palette("Blues", 4),
    width=0.65
)

# Valores encima de las barras
ax.bar_label(
    bars,
    padding=4,
    fontsize=11,
    fontweight="bold"
)

ax.set_title(
    "Distribución de conversaciones según su longitud",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Longitud de la conversación", fontsize=12)
ax.set_ylabel("Número de conversaciones", fontsize=12)

# Solo dejamos la cuadrícula horizontal
ax.grid(axis="x", visible=False)
ax.grid(axis="y", alpha=0.25)

# Eliminamos bordes superiores y laterales
sns.despine()

# Un poco de espacio arriba para las etiquetas
ax.margins(y=0.1)

plt.tight_layout()
plt.show()

In [ ]:
# 11. Crear estratos por número de caracteres de la conversación
conversation_df["text_length_bin"] = pd.cut(
    conversation_df["char_count"],
    bins=[
        0,
        2_000,
        10_000,
        25_000,
        float("inf")
    ],
    labels=[
        "compact",
        "moderate",
        "large",
        "very_large"
    ],
    include_lowest=True
)

In [ ]:
# Almacenamos el dataset de conversaciones para evitar construirlo desde un inicio
conversation_df.to_pickle(
    (PROJECT_ROOT / 'data' / 'processed' / 'conversation_dataset.pkl')
)

In [ ]:
# Carga del dataset de conversaciones (ejecutar en caso de que sea necesario)
conversation_df = pd.read_pickle(
    (PROJECT_ROOT / 'data' / 'processed' / 'conversation_dataset.pkl')
)

all_conversation_df = pd.read_pickle(
    (PROJECT_ROOT / 'data' / 'processed' / 'all_conversation_dataset.pkl')
)

### Creación de la muestra para la prueba piloto

In [ ]:
# 12. Muestreo para el conjunto de conversaciones de la prueba piloto
PILOT_DISTRIBUTION = {
    "very_short": 60,
    "short": 50,
    "medium": 60,
    "long": 30
}

pilot_parts = []

for bin_name, n_samples in PILOT_DISTRIBUTION.items():

    group = conversation_df[
        conversation_df["length_bin"] == bin_name
    ]

    if len(group) < n_samples:
        raise ValueError(
            f"No hay suficientes conversaciones en {bin_name}: "
            f"{len(group)} disponibles, {n_samples} solicitadas."
        )

    sample = group.sample(
        n=n_samples,
        random_state=42
    )

    pilot_parts.append(sample)


pilot_df = pd.concat(
    pilot_parts,
    ignore_index=True
)

pilot_df = pilot_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
# 13. Inserción del identificador a los registros de la muestra
pilot_df.insert(
    0,
    "conversation_id",
    [
        f"PILOT_{i:04d}"
        for i in range(1, len(pilot_df) + 1)
    ]
)

In [ ]:
pilot_df.head()

In [ ]:
pd.crosstab(
    pilot_df["length_bin"],
    pilot_df["text_length_bin"]
)

In [ ]:
# Visualización de casos extremos en cuanto al número de caracteres
pilot_df.sort_values(
    "char_count",
    ascending=False
)[
    [
        "conversation_id",
        "n_messages",
        "char_count",
        "length_bin",
        "topics"
    ]
].head(15)

In [ ]:
# Vamos a quitar aquellos casos extremos para no fatigar la ejecución de la prueba piloto
# Lo almacenaremos como caso de estrés
stress_cases = pilot_df[
    pilot_df["char_count"] > 50_000
].copy()

pilot_df = pilot_df[
    pilot_df["char_count"] <= 50_000
].copy()

In [ ]:
# Reemplazamos aquellas conversaciones eliminadas por otras de la misma categoría de longitud
def replace_removed_cases(
    pilot_df,
    conversation_df,
    removed_df,
    random_state=42
):
    replacements = []

    for length_bin, removed_group in removed_df.groupby("length_bin"):

        n_needed = len(removed_group)

        candidates = conversation_df[
            (conversation_df["length_bin"] == length_bin)
            & (~conversation_df["url"].isin(pilot_df["url"]))
            & (~conversation_df["url"].isin(removed_df["url"]))
            & (conversation_df["char_count"] <= 50_000)
        ]

        replacement = candidates.sample(
            n=n_needed,
            random_state=random_state
        )

        replacements.append(replacement)

    if replacements:
        pilot_df = pd.concat(
            [pilot_df, *replacements],
            ignore_index=True
        )

    return pilot_df

# Ejecutamos el reemplazamiento
pilot_df = replace_removed_cases(
    pilot_df,
    conversation_df,
    stress_cases
)

In [ ]:
pilot_df.sort_values(
    "char_count",
    ascending=False
)[
    [
        "conversation_id",
        "n_messages",
        "char_count",
        "length_bin",
        "topics"
    ]
].head(15)

In [ ]:
# Realización de chequeos para verificar la correcta construcción de la muestra piloto

# Exactamente 200 conversaciones
assert len(pilot_df) == 200

# Una fila por conversación
assert pilot_df["url"].is_unique

# Ninguna conversación redactada
assert not pilot_df["conversation_text"].str.contains(
    "<REDACTED>",
    regex=False
).any()

# Ninguna conversación vacía
assert pilot_df["conversation_text"].str.strip().ne("").all()

# Todas las conversaciones tienen ambos roles
assert conversation_df["has_user"].all()
assert conversation_df["has_assistant"].all()

# Comprobación de distribución
print(
    pilot_df["length_bin"]
    .value_counts()
)

print(
    pd.crosstab(
        pilot_df["length_bin"],
        pilot_df["text_length_bin"]
    )
)

In [ ]:
# Creamos un pequeño registro para documentar los cambios del corpus
filtering_stats = {
    "initial_conversations": 62706,
    "redacted_conversations": len(redacted_urls),
    "single_role_conversations": 1734,
    "eligible_conversations": len(conversation_df)
}

In [ ]:
# Guardar la muestra piloto

# CSV: útil para inspección manual
pilot_df.to_csv(
    (PROJECT_ROOT / 'data' / 'pilot' / 'pilot_dataset.csv'),
    index=False,
    encoding="utf-8"
)

# Pickle: recomendable para reutilizar desde Python
pilot_df.to_pickle(
    (PROJECT_ROOT / 'data' / 'pilot' / 'pilot_dataset.pkl')
)

print(f"Pilot dataset guardado: {len(pilot_df)} conversaciones")

In [ ]:
# Carga del dataset de la muestra piloto (en caso de que sea necesario)
pilot_df = pd.read_pickle(
    (PROJECT_ROOT / 'data' / 'pilot' / 'pilot_dataset.pkl')
)

### Creación de la muestra de anotación humana para la prueba piloto

In [ ]:
# 1. Selección de la muestra de anotación humana

HUMAN_SAMPLE_DISTRIBUTION = {
    "very_short": 20,
    "short": 18,
    "medium": 22,
    "long": 15
}

HUMAN_RANDOM_STATE = 123

human_parts = []

for length_bin, n_samples in HUMAN_SAMPLE_DISTRIBUTION.items():

    subset = pilot_df[
        pilot_df["length_bin"] == length_bin
    ]

    if len(subset) < n_samples:
        raise ValueError(
            f"No hay suficientes conversaciones en '{length_bin}'. "
            f"Disponibles: {len(subset)}, solicitadas: {n_samples}"
        )

    sample = subset.sample(
        n=n_samples,
        random_state=HUMAN_RANDOM_STATE
    )

    human_parts.append(sample)


human_sample_df = pd.concat(
    human_parts,
    ignore_index=True
)

# Mezclamos para que no aparezcan ordenadas por longitud
human_sample_df = human_sample_df.sample(
    frac=1,
    random_state=HUMAN_RANDOM_STATE
).reset_index(drop=True)


# Comprobaciones
assert len(human_sample_df) == 75
assert human_sample_df["conversation_id"].is_unique
assert set(human_sample_df["conversation_id"]).issubset(
    set(pilot_df["conversation_id"])
)

print(
    human_sample_df["length_bin"]
    .value_counts()
)

In [ ]:
# 2. Guardado de la muestra de anotación manual
human_sample_df.to_csv(
    (PROJECT_ROOT / 'data' / 'pilot' / 'human_sample_conversations.csv'),
    index=False,
    encoding="utf-8"
)

human_sample_df.to_pickle(
    (PROJECT_ROOT / 'data' / 'pilot' / 'human_sample_conversations.pkl')
)

print(
    f"Muestra humana guardada: {len(human_sample_df)} conversaciones"
)

In [ ]:
# Carga del dataset de la muestra de anotación manual (en caso de que sea necesario)
human_sample_df = pd.read_pickle(
    (PROJECT_ROOT / 'data' / 'pilot' / 'human_sample_conversations.pkl')
)

In [ ]:
# 3. Creación de la plantilla de anotación humana
BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

annotation_rows = []

for _, row in human_sample_df.iterrows():

    for bias in BIASES:

        annotation_rows.append({
            "conversation_id": row["conversation_id"],
            "bias": bias,

            # Campos que rellenará el anotador humano
            "human_label": "",
            "human_confidence": "",
            "trigger_turns": "",
            "evidence_turns": "",
            "notes": ""
        })


human_annotations_df = pd.DataFrame(annotation_rows)

assert len(human_annotations_df) == 75 * 4

human_annotations_df.to_csv(
    (PROJECT_ROOT / 'data' / 'processed' / 'pilot' / 'human_annotations.csv'),
    index=False,
    encoding="utf-8"
)

print(
    f"Plantilla creada: {len(human_annotations_df)} anotaciones"
)

### Estimador de tokens y costes

In [ ]:
%pip install tiktoken

In [ ]:
import tiktoken
from pathlib import Path

In [ ]:
# Carga de los prompts
PROMPT_DIR = PROMPTS_DIR

PROMPT_FILES = {
    "confirmation_bias": PROMPT_DIR / "confirmation_bias.txt",
    "anchoring_bias": PROMPT_DIR / "anchoring_bias.txt",
    "framing_effect": PROMPT_DIR / "framing_effect.txt",
    "availability_bias": PROMPT_DIR / "availability_bias.txt",
}

In [ ]:
from bias_detector.annotation_utils import load_prompt_templates
PROMPTS = load_prompt_templates(PROMPT_FILES)


In [ ]:
# Estimación con un tokenizador que va a ser "o200k_base"
encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    if not isinstance(text, str):
        text = str(text)

    return len(
        encoding.encode(
            text,
            disallowed_special=() # HAY QUE REVISAR LOS TOKENS ESPECIALES MARCADOS POR <>
        )
    )

In [ ]:
# Tokens fijos de cada prompt: queremos saber cúanto cuesta la parte que repetimos en todas las conversaciones
prompt_token_stats = []

for bias, prompt in PROMPTS.items():

    # Quitamos el placeholder porque luego introduciremos
    # la conversación real por separado.
    prompt_without_conversation = prompt.replace(
        "{{CONVERSATION}}",
        ""
    )

    tokens = count_tokens(
        prompt_without_conversation
    )

    prompt_token_stats.append({
        "bias": bias,
        "prompt_tokens": tokens
    })


prompt_tokens_df = pd.DataFrame(
    prompt_token_stats
)

print(prompt_tokens_df)

print(
    "\nTokens totales de los cuatro prompts:",
    prompt_tokens_df["prompt_tokens"].sum()
)

In [ ]:
# Contamos tokens de cada conversación del piloto
pilot_df["conversation_tokens"] = (
    pilot_df["conversation_text"]
    .apply(count_tokens)
)

In [ ]:
# Miramos la distribución
print(
    pilot_df["conversation_tokens"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
# Miramos los casos más grandes
print(
    pilot_df[
        [
            "conversation_id",
            "n_messages",
            "char_count",
            "conversation_tokens"
        ]
    ]
    .sort_values(
        "conversation_tokens",
        ascending=False
    )
    .head(15)
)

In [ ]:
# Calculamos ahora los tokens reales de entrada por conversación 
# Cada conversación se utiliza cuatro veces, por tanto, también los tokens de conversación

TOTAL_FIXED_PROMPT_TOKENS = (
    prompt_tokens_df["prompt_tokens"].sum()
)

N_BIASES = len(PROMPTS)


pilot_df["input_tokens_one_annotator"] = (
    TOTAL_FIXED_PROMPT_TOKENS
    +
    N_BIASES * pilot_df["conversation_tokens"]
)

In [ ]:
# Miramos la distribución
print(
    pilot_df["input_tokens_one_annotator"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
# Tenemos que estimar también la salida, la cual no sabemos cuántos tokens utilizará realmente
# Son tokens por anotación, no por conversación
OUTPUT_SCENARIOS = {
    "low": 100,
    "expected": 175,
    "high": 300
}

In [ ]:
for scenario, output_per_annotation in OUTPUT_SCENARIOS.items():

    pilot_df[
        f"output_tokens_{scenario}_one_annotator"
    ] = (
        output_per_annotation * N_BIASES
    )

In [ ]:
# Tabla de precios de los LLMs
PRICING = {
    "gpt-5.6-luna": {
        "input": 0.20,
        "output": 1.20
    },
    "gpt-5.6-terra": {
        "input": 2.00,
        "output": 12.00
    },
    "gpt-5.6-sol": {
        "input": 4.00,
        "output": 20.00
    }
}

In [ ]:
# Función de coste
def calculate_cost(
    input_tokens,
    output_tokens,
    input_price_per_million,
    output_price_per_million
):
    input_cost = (
        input_tokens
        / 1_000_000
        * input_price_per_million
    )

    output_cost = (
        output_tokens
        / 1_000_000
        * output_price_per_million
    )

    return input_cost + output_cost

In [ ]:
# Estimación del piloto completo de 200 conversaciones
def estimate_dataset_cost(
    dataset,
    model_name,
    output_tokens_per_annotation=175,
    n_annotators=1
):

    pricing = PRICING[model_name]

    total_input_tokens = (
        dataset[
            "input_tokens_one_annotator"
        ].sum()
        * n_annotators
    )

    total_output_tokens = (
        len(dataset)
        * N_BIASES
        * output_tokens_per_annotation
        * n_annotators
    )

    input_cost = (
        total_input_tokens
        / 1_000_000
        * pricing["input"]
    )

    output_cost = (
        total_output_tokens
        / 1_000_000
        * pricing["output"]
    )

    return {
        "model": model_name,
        "conversations": len(dataset),
        "annotators": n_annotators,
        "input_tokens": int(total_input_tokens),
        "output_tokens": int(total_output_tokens),
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": input_cost + output_cost
    }

In [ ]:
# Coste de un anotador
pilot_estimates = []

for model_name in PRICING:

    estimate = estimate_dataset_cost(
        dataset=pilot_df,
        model_name=model_name,
        output_tokens_per_annotation=175,
        n_annotators=1
    )

    pilot_estimates.append(
        estimate
    )


pilot_cost_df = pd.DataFrame(
    pilot_estimates
)

print(
    pilot_cost_df.to_string(
        index=False
    )
)

In [ ]:
# Si se utilizasen dos anotadores
two_annotator_estimates = []

for model_name in PRICING:

    estimate = estimate_dataset_cost(
        dataset=pilot_df,
        model_name=model_name,
        output_tokens_per_annotation=175,
        n_annotators=2
    )

    two_annotator_estimates.append(
        estimate
    )


pd.DataFrame(
    two_annotator_estimates
)

In [ ]:
# Lo interesante es estimar las 9679 conversaciones con las que contamos

# Aplicamos el conteo de tokens
conversation_df["conversation_tokens"] = (
    conversation_df["conversation_text"]
    .apply(count_tokens)
)

# Multliplicamos por el número de sesgos que tenemos
conversation_df["input_tokens_one_annotator"] = (
    TOTAL_FIXED_PROMPT_TOKENS
    +
    N_BIASES
    * conversation_df["conversation_tokens"]
)

# Obtenemos la estimación de los precios
full_estimates = []

for model_name in PRICING:

    estimate = estimate_dataset_cost(
        dataset=conversation_df,
        model_name=model_name,
        output_tokens_per_annotation=175,
        n_annotators=1
    )

    full_estimates.append(
        estimate
    )


full_cost_df = pd.DataFrame(
    full_estimates
)

print(
    full_cost_df.to_string(
        index=False
    )
)

In [ ]:
# Si recuperamos las 59199 conversaciones iniciales

# Aplicamos el conteo de tokens
all_conversation_df["conversation_tokens"] = (
    all_conversation_df[
        "conversation_text"
    ].apply(count_tokens)
)

# Multliplicamos por el número de sesgos que tenemos
all_conversation_df[
    "input_tokens_one_annotator"
] = (
    TOTAL_FIXED_PROMPT_TOKENS
    +
    N_BIASES
    * all_conversation_df[
        "conversation_tokens"
    ]
)

# Obtenemos la estimación de los precios
full_estimates = []

for model_name in PRICING:

    estimate = estimate_dataset_cost(
        dataset=all_conversation_df,
        model_name=model_name,
        output_tokens_per_annotation=175,
        n_annotators=1
    )

    full_estimates.append(
        estimate
    )


full_cost_df = pd.DataFrame(
    full_estimates
)

print(
    full_cost_df.to_string(
        index=False
    )
)

In [ ]:
# Hagamos una tabla completa de forma automática
def compare_models_and_sizes(
    datasets,
    output_tokens_per_annotation=175
):

    rows = []

    for dataset_name, dataset in datasets.items():

        for model_name in PRICING:

            result = estimate_dataset_cost(
                dataset=dataset,
                model_name=model_name,
                output_tokens_per_annotation=output_tokens_per_annotation,
                n_annotators=1
            )

            result["dataset"] = dataset_name

            rows.append(result)

    columns = [
        "dataset",
        "model",
        "conversations",
        "input_tokens",
        "output_tokens",
        "input_cost_usd",
        "output_cost_usd",
        "total_cost_usd"
    ]

    return pd.DataFrame(rows)[columns]

In [ ]:
# Generamos la tabla con los datasets seleccionados
datasets = {
    "pilot_200": pilot_df,
    "clean_corpus": conversation_df,
    "all_corpus": all_conversation_df
}

cost_comparison_df = compare_models_and_sizes(
    datasets
)

print(
    cost_comparison_df.to_string(
        index=False
    )
)

In [ ]:
# Vamos a examinar ahora qué conversación nos está saliendo más cara
MODEL_TO_INSPECT = "gpt-5.6-luna"

price = PRICING[MODEL_TO_INSPECT]

pilot_df["estimated_input_cost"] = (
    pilot_df["input_tokens_one_annotator"]
    / 1_000_000
    * price["input"]
)

pilot_df[
    [
        "conversation_id",
        "n_messages",
        "conversation_tokens",
        "input_tokens_one_annotator",
        "estimated_input_cost"
    ]
].sort_values(
    "estimated_input_cost",
    ascending=False
).head(20)

In [ ]:
# Otra estimación especialmente útil sería calcular el coste por 1000 conversaciones
def estimated_cost_per_1000(
    dataset,
    model_name,
    output_tokens_per_annotation=175
):

    result = estimate_dataset_cost(
        dataset=dataset,
        model_name=model_name,
        output_tokens_per_annotation=output_tokens_per_annotation
    )

    return (
        result["total_cost_usd"]
        / len(dataset)
        * 1000
    )

In [ ]:
for model in PRICING:

    cost = estimated_cost_per_1000(
        pilot_df,
        model
    )

    print(
        f"{model}: "
        f"${cost:.2f} / 1,000 conversations"
    )

Hasta aquí, el estudio sobre la estimación de cuánto nos puede costar la anotación de los conjuntos de la prueba piloto y los corpus.

### Estudio detallado de los tokens especiales del corpus

In [ ]:
conversation_df["conversation_tokens"].describe()

In [ ]:
all_conversation_df["conversation_tokens"].describe()

In [ ]:
# Inventario de todos los tokens detectados
SPECIAL_TOKEN_PATTERN = r"<[^<>]+>"

token_occurrences = (
    df_work["plain_text"]
    .dropna()
    .str.findall(SPECIAL_TOKEN_PATTERN)
    .explode()
    .dropna()
)

special_token_counts = (
    token_occurrences
    .value_counts()
    .rename_axis("token")
    .reset_index(name="count")
)

special_token_counts.head(30)

In [ ]:
print(
    "Tokens distintos:",
    special_token_counts["token"].nunique()
)

print(
    "Ocurrencias totales:",
    special_token_counts["count"].sum()
)

In [ ]:
# Medición de problema a nivel de mensaje
# Vemos si <REDACTED> aparece predominantemente en mensajes humanos o también en las respuestas del asistente
df_work["special_tokens"] = (
    df_work["plain_text"]
    .fillna("")
    .str.findall(SPECIAL_TOKEN_PATTERN)
)

df_work["n_special_tokens"] = (
    df_work["special_tokens"]
    .str.len()
)

df_work["has_special_token"] = (
    df_work["n_special_tokens"] > 0
)

df_work["has_redacted"] = (
    df_work["plain_text"]
    .fillna("")
    .str.contains(
        "<REDACTED>",
        regex=False
    )
)

In [ ]:
print(
    df_work[
        "has_redacted"
    ].value_counts()
)

print(
    pd.crosstab(
        df_work["role"],
        df_work["has_redacted"]
    )
)

In [ ]:
# A nivel de conversación
redaction_stats = (
    df_work
    .groupby("url")
    .agg(
        n_messages=(
            "plain_text",
            "size"
        ),
        affected_messages=(
            "has_redacted",
            "sum"
        ),
        redacted_tokens=(
            "n_special_tokens",
            "sum"
        ),
        has_redacted=(
            "has_redacted",
            "any"
        )
    )
    .reset_index()
)

In [ ]:
redaction_stats[
    "affected_message_ratio"
] = (
    redaction_stats["affected_messages"]
    / redaction_stats["n_messages"]
)

In [ ]:
redaction_stats[
    redaction_stats["has_redacted"]
][
    [
        "affected_messages",
        "redacted_tokens",
        "affected_message_ratio"
    ]
].describe(
    percentiles=[
        .25,
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)

In [ ]:
# Intensidad de redacción
def redaction_level(row):

    if row["affected_messages"] == 0:
        return "none"

    if (
        row["affected_messages"] == 1
        and row["redacted_tokens"] == 1
    ):
        return "minimal"

    if row["affected_message_ratio"] <= 0.10:
        return "low"

    if row["affected_message_ratio"] <= 0.30:
        return "moderate"

    return "high"


redaction_stats["redaction_level"] = (
    redaction_stats.apply(
        redaction_level,
        axis=1
    )
)

In [ ]:
redaction_stats[
    "redaction_level"
].value_counts()

In [ ]:
# Estudiamos también el contexto, no solo las estadísticas
def extract_redacted_context(
    text,
    window=150
):

    if pd.isna(text):
        return []

    contexts = []

    for match in re.finditer(
        re.escape("<REDACTED>"),
        text
    ):

        start = max(
            0,
            match.start() - window
        )

        end = min(
            len(text),
            match.end() + window
        )

        contexts.append(
            text[start:end]
        )

    return contexts

In [ ]:
redacted_messages = (
    df_work[
        df_work["has_redacted"]
    ][
        [
            "url",
            "message_index",
            "role",
            "plain_text"
        ]
    ]
    .copy()
)

redacted_messages[
    "redacted_contexts"
] = (
    redacted_messages["plain_text"]
    .apply(extract_redacted_context)
)

In [ ]:
redacted_messages.iloc[112]["redacted_contexts"]

2º parte del análisis

In [ ]:
# Separar placeholders de anonimización del HTML
PLACEHOLDER_PATTERN = r"<[A-Z][A-Z0-9_]*>"

placeholder_occurrences = (
    df_work["plain_text"]
    .fillna("")
    .str.findall(PLACEHOLDER_PATTERN)
    .explode()
    .dropna()
)

placeholder_counts = (
    placeholder_occurrences
    .value_counts()
    .rename_axis("token")
    .reset_index(name="count")
)

placeholder_counts.head(50)

In [ ]:
# Repetimos el análisis de la intensidad de redacción solo sobre conversaciones completas
eligible_ids = set(
    conversation_df.loc[
        conversation_df["has_user"]
        & conversation_df["has_assistant"],
        "url"
    ]
)

print(len(eligible_ids))

In [ ]:
eligible_messages_df = df_work[
    df_work["url"].isin(eligible_ids)
].copy()

In [ ]:
# Distinguir los tres placeholders principales
eligible_messages_df["n_redacted"] = (
    eligible_messages_df["plain_text"]
    .fillna("")
    .str.count(
        re.escape("<REDACTED>")
    )
)

eligible_messages_df["n_url_placeholder"] = (
    eligible_messages_df["plain_text"]
    .fillna("")
    .str.count(
        re.escape("<URL>")
    )
)

eligible_messages_df["n_datetime_placeholder"] = (
    eligible_messages_df["plain_text"]
    .fillna("")
    .str.count(
        re.escape("<DATE_TIME>")
    )
)

In [ ]:
eligible_messages_df["has_redacted"] = (
    eligible_messages_df["n_redacted"] > 0
)

eligible_messages_df["has_url_placeholder"] = (
    eligible_messages_df["n_url_placeholder"] > 0
)

eligible_messages_df["has_datetime_placeholder"] = (
    eligible_messages_df["n_datetime_placeholder"] > 0
)

In [ ]:
# Medir intensidad de <REDACTED> con mñas precisión
redaction_stats = (
    eligible_messages_df
    .groupby("url")
    .agg(
        n_messages=("plain_text", "size"),
        affected_messages=("has_redacted", "sum"),
        n_redacted=("n_redacted", "sum")
    )
    .reset_index()
)

redaction_stats["affected_message_ratio"] = (
    redaction_stats["affected_messages"]
    / redaction_stats["n_messages"]
)

In [ ]:
redaction_stats[
    redaction_stats["n_redacted"] > 0
][
    [
        "n_messages",
        "affected_messages",
        "n_redacted",
        "affected_message_ratio"
    ]
].describe(
    percentiles=[
        .25, .50, .75, .90, .95, .99
    ]
)

In [ ]:
# Candidatos recuperables (un único <REDACTED> en un único mensaje)
minimal_candidates = redaction_stats[
    (redaction_stats["n_redacted"] == 1)
    &
    (redaction_stats["affected_messages"] == 1)
].copy()

print(len(minimal_candidates))

In [ ]:
# Unir características conversacionales
minimal_candidates = minimal_candidates.merge(
    conversation_df[
        [
            "url",
            "n_messages",
            "char_count",
            "length_bin"
        ]
    ],
    on="url",
    how="left"
)

In [ ]:
sample_minimal = (
    minimal_candidates
    .sample(
        n=min(
            100,
            len(minimal_candidates)
        ),
        random_state=2026
    )
)

In [ ]:
sample_minimal

In [ ]:
# Dimensión extra: quien contiene <REDACTED>
redaction_by_conversation_role = (
    eligible_messages_df[
        eligible_messages_df["has_redacted"]
    ]
    .groupby(
        [
            "url",
            "role"
        ]
    )["n_redacted"]
    .sum()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

redaction_by_conversation_role.head()

In [ ]:
def redaction_location(row):

    user_count = row.get("user", 0)
    llm_count = row.get("llm", 0)

    if user_count > 0 and llm_count > 0:
        return "both"

    if user_count > 0:
        return "user_only"

    if llm_count > 0:
        return "llm_only"

    return "none"


redaction_by_conversation_role[
    "redaction_location"
] = (
    redaction_by_conversation_role.apply(
        redaction_location,
        axis=1
    )
)

redaction_by_conversation_role[
    "redaction_location"
].value_counts()

Vamos a guardar estas conversaciones candidatas a ser recuperadas, en caso de que sean necesarias a volver tras la anotación masiva del corpus base.

In [ ]:
# Añadimos localización del token por rol
minimal_candidates = minimal_candidates.merge(
    redaction_by_conversation_role[
        [
            "url",
            "redaction_location"
        ]
    ],
    on="url",
    how="left",
    validate="one_to_one"
)

In [ ]:
# Añadimos características generales de la conversación desde
# la tabla conversacional completa
minimal_candidates = minimal_candidates.merge(
    conversation_df[
        [
            "url",
            "n_messages",
            "char_count",
            "length_bin",
            "conversation_text"
        ]
    ],
    on="url",
    how="left",
    validate="one_to_one"
)

In [ ]:
minimal_candidates = minimal_candidates[
    [
        "url",
        "n_messages",
        "affected_messages",
        "n_redacted",
        "affected_message_ratio",
        "redaction_location",
        "conversation_text"
    ]
].copy()

# faltan char_count y length_bin, habrá que computarlas posteriormente

In [ ]:
print(minimal_candidates.shape)

assert len(minimal_candidates) == 3519
assert minimal_candidates["url"].is_unique

In [ ]:
print(
    minimal_candidates[
        "redaction_location"
    ].value_counts(dropna=False)
)

In [ ]:
# Guardado
PROJECT_ROOT = _locate_repo()

REDACTION_DIR = (
    PROJECT_ROOT
    / "data"
    / "redaction_analysis"
)

REDACTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
minimal_candidates[
    "recovery_candidate_rule"
] = "ONE_REDACTED_ONE_MESSAGE"

In [ ]:
MINIMAL_REDACTED_PKL = (
    REDACTION_DIR
    / "minimal_redacted_candidates.pkl"
)

minimal_candidates.to_pickle(
    MINIMAL_REDACTED_PKL
)

print(
    "Candidatos guardados:",
    len(minimal_candidates)
)